In [1]:
import xarray as xr

import os
from datetime import datetime
import glob
file = glob.glob("/Users/Indhuja/Desktop/sc-radiosonde/SCRX_Radiosonde_CPEXAW_20210914_2003.nc")
print(file[0])

/Users/Indhuja/Desktop/sc-radiosonde/SCRX_Radiosonde_CPEXAW_20210914_2003.nc


In [2]:
with xr.open_dataset("/Users/Indhuja/Desktop/sc-radiosonde/CPEXAW-DROPSONDE_D20210806_193025_PQC.nc",decode_cf=False) as ds:
# with xr.open_dataset("/Users/Indhuja/Desktop/sc-radiosonde/SCRX_Radiosonde_CPEXAW_win_20210819_2312.nc", decode_cf=True) as ds:
    print(ds.data_vars)
    print("-----------")
    print(ds.attrs)
    print("-----------")
    for key in ds.attrs:
            print(key)
    time = ds['time'].values
    print(ds['launch_time'])
# for v in time:
#     print(v)

    
    

Data variables:
    trajectory      |S1 ...
    launch_time     int32 ...
    pres            (time) float32 ...
    tdry            (time) float32 ...
    dp              (time) float32 ...
    rh              (time) float32 ...
    u_wind          (time) float32 ...
    v_wind          (time) float32 ...
    w_wind          (time) float32 ...
    wspd            (time) float32 ...
    wdir            (time) float32 ...
    dz              (time) float32 ...
    mr              (time) float32 ...
    vt              (time) float32 ...
    theta           (time) float32 ...
    theta_e         (time) float32 ...
    theta_v         (time) float32 ...
    lat             (time) float32 ...
    lon             (time) float32 ...
    alt             (time) float32 ...
    gpsalt          (time) float32 ...
    reference_time  (obs) int32 ...
    reference_pres  (obs) float32 ...
    reference_tdry  (obs) float32 ...
    reference_rh    (obs) float32 ...
    reference_wspd  (obs) float32 .

In [106]:
file = glob.glob("/Users/Indhuja/Desktop/sc-radiosonde/SCRX_Radiosonde_CPEXAW_win_20210914_2003.txt")
input_file = os.path.basename("/Users/Indhuja/Desktop/sc-radiosonde/SCRX_Radiosonde_CPEXAW_win_20210914_2003.txt")

# start_date = input_file.split("/")[5].split("_")[4]
# start_time = input_file.split("/")[5].split("_")[5].split(".")[0]
# print(start_time)

date_str, time_str = input_file.split("_")[-2:]
print(date_str)
time_str = time_str.replace(".txt", "")
print(time_str)

datetime_str = f"{date_str}_{time_str}"
print(datetime_str)
# Parsing the datetime string
original_datetime = datetime.strptime(datetime_str, "%Y%m%d_%H%M")
print(original_datetime)

# Formatting to the desired output format
formatted_datetime = original_datetime.strftime("%Y-%m-%dT%H:%M:%S")
print(formatted_datetime)

20210914
2003
20210914_2003
2021-09-14 20:03:00
2021-09-14T20:03:00


In [163]:
import pandas as pd
lines = []
# for line in file[0]:
#     line = line.decode()
#     lines.append(line.split())
# lines
# print(file)
with open(file[0],encoding = "ISO-8859-1") as f:
    lines=f.readlines()
# for line in lines:
#     print(line)
df = pd.read_table(file[0],encoding = "ISO-8859-1")
print(df.columns.values)
# df1 = df.rename(columns={'Time [sec]':'Time', 'T [°C]':'Temp', 'U [%]':'RH', 'Lat [°]':'Lat'})
# df1 = df.iloc[:, ]

column_name_changes = {
    'Time [sec]': 'Time',
    'T [°C]': 'Temp',
    'U [%]': 'RH',
    'Lon [°]   ': 'Lon',
    'Lat [°]  ': 'Lat',
    'Altitude [m]': 'Alt',
    'Dew [°C]': 'DP'
}

df.rename(columns=column_name_changes, inplace=True)
valid_columns = list(column_name_changes.values())
print(valid_columns)
df = df[valid_columns]
df.replace('-----', pd.NA, inplace=True)
df.dropna(inplace=True)
print(df)

start_time = pd.to_datetime(formatted_datetime)
df['Datetime'] = start_time + pd.to_timedelta(df['Time'], unit='s')
print(df)

['Time [sec]' 'P [h Pa]' 'T [°C]' 'U [%]' 'Wsp [m/s]' 'Wdir [°]'
 'Lon [°]   ' 'Lat [°]  ' 'Altitude [m]' "Geo Pot [m']" 'MRI   ' 'RI   '
 'Dew [°C]' 'Vi Te [°C]' 'Rs [m/s]' 'D [kg/m3]  ' 'Azimuth [°]'
 'Elevation [°]' 'Range [m]']
['Time', 'Temp', 'RH', 'Lon', 'Lat', 'Alt', 'DP']
      Time   Temp     RH        Lon        Lat      Alt    DP
0        0  29.50  71    -64.831754  17.762489      0.0  23.7
1        1  29.51  72    -64.831807  17.762449      6.5  23.9
2        2  29.53  73    -64.831860  17.762408     13.0  24.1
3        3  29.54  73    -64.831913  17.762368     19.4  24.2
4        4  29.55  74    -64.831965  17.762328     25.9  24.4
...    ...    ...    ...        ...        ...      ...   ...
3364  3364 -73.25  59    -64.793124  17.900495  15707.5 -76.6
3365  3365 -73.23  59    -64.793056  17.900533  15711.5 -76.6
3366  3366 -73.22  58    -64.792987  17.900570  15715.5 -76.7
3367  3367 -73.20  58    -64.792919  17.900608  15719.5 -76.7
3368  3368 -73.19  57    -64.792850 

In [35]:
import numpy as np
import xarray as xr
from datetime import datetime, timedelta

def stringToDateTime(date_str, time_str):
  date = datetime.strptime(date_str, '%Y%m%d')
  time = datetime.strptime(time_str, '%H%M%S')
  return datetime.combine(date.date(), time.time())

def addDelta(dateTime, s):
  delta = timedelta(milliseconds=s*1000)
  combined_date_time = (dateTime + delta)
  return combined_date_time.isoformat(sep='T', timespec='auto')

date = "20210806"
time = "193025"

# date = "20210914"
# time = "2003"

base_time = stringToDateTime(date, time)
print(base_time)

# open dataset.
with xr.open_dataset("/Users/Indhuja/Desktop/sc-radiosonde/CPEXAW-DROPSONDE_D20210806_193025_PQC.nc", decode_cf=False) as ds:
    rh = ds['rh'].values # relative humidity
    dp = ds['dp'].values # dew point
    tdry = ds['tdry'].values # temp dry???
    lat = ds['lat'].values
    lon = ds['lon'].values
    alt = ds['alt'].values
    timesec = ds['time'].values

print(timesec)
timestr = np.vectorize(addDelta)(base_time, timesec)
print(timestr)
time = np.array(timestr, dtype='datetime64[s]').astype(np.int64)
print(time)

n_time = np.array([], dtype=np.int64)
n_time = np.append(n_time, time)
epoch = np.min(n_time)
n_time = (n_time - epoch).astype(np.int32)
print(epoch)
with np.printoptions(threshold=np.inf):
  print(timestr)
  print(n_time)

2021-08-06 19:30:25
[6.44650024e+02 6.44109985e+02 6.43859985e+02 ... 6.09999895e-01
 3.59999895e-01 1.09999895e-01]
['2021-08-06T19:41:09.650024' '2021-08-06T19:41:09.109985'
 '2021-08-06T19:41:08.859985' ... '2021-08-06T19:30:25.610000'
 '2021-08-06T19:30:25.360000' '2021-08-06T19:30:25.110000']
[1628278869 1628278869 1628278868 ... 1628278225 1628278225 1628278225]
1628278225
['2021-08-06T19:41:09.650024' '2021-08-06T19:41:09.109985'
 '2021-08-06T19:41:08.859985' '2021-08-06T19:41:08.609985'
 '2021-08-06T19:41:08.359985' '2021-08-06T19:41:08.109985'
 '2021-08-06T19:41:07.859985' '2021-08-06T19:41:07.609985'
 '2021-08-06T19:41:07.359985' '2021-08-06T19:41:07.109985'
 '2021-08-06T19:41:06.859985' '2021-08-06T19:41:06.609985'
 '2021-08-06T19:41:06.359985' '2021-08-06T19:41:06.109985'
 '2021-08-06T19:41:05.859985' '2021-08-06T19:41:05.609985'
 '2021-08-06T19:41:05.359985' '2021-08-06T19:41:05.109985'
 '2021-08-06T19:41:04.859985' '2021-08-06T19:41:04.609985'
 '2021-08-06T19:41:04.359985

In [215]:
data = pd.read_csv("/Users/Indhuja/Desktop/pr-radiosonde/PR_Radiosonde_CPEXAW_qc_uprm001_20210824.csv",skiprows=118)
print(data.columns)

column_name_changes = {
    'sec': 'Time',
    'deg C': 'Temp',
    '%': 'RH',
    'deg.1': 'Lat',
    'deg.2': 'Lon',
    'm': 'Alt',
    'deg C.1': 'DP',
    'm/s.3': 'Ascent'
}

data.rename(columns=column_name_changes, inplace=True)
valid_columns = list(column_name_changes.values())
print(valid_columns)
data = data[valid_columns]


initial_altitude = 27
data['Alt'] = initial_altitude + (data['Time'] * data['Ascent'])
print(data['Alt'].to_list())
print(data)

# df = pd.read_csv("/Users/Indhuja/Desktop/pr-radiosonde/PR_Radiosonde_CPEXAW_qc_uprm001_20210824.csv", header=None, names=range(15))
# print(df[1].values[1])
# year = df[1].values[1]
# month = df[1].values[2]
# day = df[1].values[3]
# hour = df[1].values[4]
# minute = df[1].values[5]
# second = df[1].values[6]

# date_time = pd.to_datetime(f'{year}{month:02s}{day:02s} {hour:02s}:{minute:02s}:{second:02s}')

# # Format the date and time
# formatted_date = date_time.strftime('%Y%m%d')
# formatted_time = date_time.strftime('%H:%M:%S')

# # Print the results
# print(f'Formatted Date: {formatted_date}')
# print(f'Formatted time: {formatted_time}')

Index(['Units', 'sec', 'mb', 'deg C', '%', 'm/s', 'deg', 'deg.1', 'deg.2', 'm',
       'm.1', 'deg C.1', 'm/s.1', 'm/s.2', 'm/s.3'],
      dtype='object')
['Time', 'Temp', 'RH', 'Lat', 'Lon', 'Alt', 'DP', 'Ascent']
[27.0, 32.3, 37.9, 43.89, 50.08, 56.4, 62.82, 69.35, 76.03999999999999, 82.71000000000001, 89.0, 94.65, 99.6, 104.09, 109.74000000000001, 116.4, 123.96, 131.55, 136.44, 138.53, 142.08, 146.45999999999998, 153.27, 161.64000000000001, 171.0, 179.36, 187.10999999999999, 194.44, 201.29, 208.2, 215.17000000000002, 221.56, 227.31, 233.04, 239.8, 247.32, 256.4, 265.64, 274.65, 283.8, 292.68, 302.09999999999997, 312.95, 323.56, 333.90000000000003, 342.56, 348.95, 353.88, 359.22, 366.5, 375.84, 385.28, 392.70000000000005, 394.73999999999995, 388.9, 368.43, 363.4, 385.8, 412.52000000000004, 439.92, 450.99, 450.04, 436.86, 437.04, 444.52, 454.11, 464.5, 473.59, 480.59999999999997, 485.44, 488.02000000000004, 500.48, 508.25, 515.28, 519.1700000000001, 520.6, 521.0999999999999, 523.09999

In [33]:
from boto3 import client as boto_client
import pandas as pd

s3_url = "s3://ghrc-fcx-field-campaigns-szg/CPEX-AW/instrument-raw-data/radiosonde/SCRX_Radiosonde_CPEXAW_win_20210914_2003.txt"
## Open data file
bucket_name = s3_url.split("/")[2]
key = s3_url.split(f"{bucket_name}/")[-1] # need key without starting /
s3 = boto_client('s3')
fileobj = s3.get_object(Bucket=bucket_name, Key=key)
file = fileobj['Body'].read().decode('ISO-8859-1')
print(file)

from io import StringIO
file = StringIO(file)

# modified_lines = []
# for line in file:
#     modified_lines.append(line)

# print(modified_lines)

# data = pd.DataFrame(modified_lines)
data = pd.read_table(file)

Time [sec]	P [h Pa]	T [°C]	U [%]	Wsp [m/s]	Wdir [°]	Lon [°]   	Lat [°]  	Altitude [m]	Geo Pot [m']	MRI   	RI   	Dew [°C]	Vi Te [°C]	Rs [m/s]	D [kg/m3]  	Azimuth [°]	Elevation [°]	Range [m]
0         	1009.3  	29.50 	71   	1.0      	080     	-64.831754	17.762489	0.0         	0.0         	378.2 	378.2	23.7    	32.8      	0.0     	1.161977   	180        	18           	0        
1         	1008.6  	29.51 	72   	1.3      	072     	-64.831807	17.762449	6.5         	6.5         	380.3 	379.3	23.9    	32.9      	6.5     	1.161089   	53         	43           	10       
2         	1007.8  	29.53 	73   	1.6      	067     	-64.831860	17.762408	13.0        	12.9        	382.5 	380.5	24.1    	32.9      	6.5     	1.160202   	52         	43           	19       
3         	1007.1  	29.54 	73   	2.0      	064     	-64.831913	17.762368	19.4        	19.4        	384.7 	381.6	24.2    	33.0      	6.5     	1.159315   	52         	42           	29       
4         	1006.4  	29.55 	74   	2.3      	061     	-64

In [34]:
timesec = data['Time [sec]']

time = np.array(timesec, dtype='datetime64[s]').astype(np.int64)
print(time)

n_time = np.array([], dtype=np.int32)
n_time = np.append(n_time, time)
epoch = np.max(n_time)
print(epoch)
n_time = (n_time - epoch).astype(np.int32)
print(n_time)

[   0    1    2 ... 3629 3630 3631]
0
[   0    1    2 ... 3629 3630 3631]
